#  Clasificación en Cascada desde Parámetros Hiperfinos

Este notebook clasifica la clase cristaloquímica a partir del vector hiperfino comprimido $\vec{P} \in \mathbb{R}^{15}$.

## Trade-off: Pérdida de Información Continua vs. Precisión

El vector $\vec{P} \in \mathbb{R}^{15}$ es una representación comprimida del espectro completo.
Al colapsar la forma de línea ($N$ puntos) en 15 parámetros hiperfinos,
se asume que el perfil Lorentziano es el único portador de información relevante.

**Casos donde esta suposición falla:**
- Distribuciones de $B_{HF}$ (broadening magnético en materiales no cristalinos)
- Asimetrías de línea no-Lorentzianas (efecto Goldanskii-Karyagin)
- Relajación superparamagnética (nanopartículas)

Comparar el macro-F1 de este notebook vs. Notebook 2 para cuantificar
numéricamente la pérdida de información introducida por la compresión paramétrica.

## Notación Tensorial
- **Entrada**: $\vec{P} \in \mathbb{R}^{B \times 15}$ — vector hiperfino estandarizado por Z-score
- **Salida**: $\hat{y} \in \mathbb{R}^{B \times 8}$ — logits sobre las 8 clases químicas

### Diagrama ASCII del MLP Ligero:
```
P: (B, 15)
    |
    v
[Linear(15->64) + BatchNorm1d + ReLU + Dropout(0.3)]  --> (B, 64)
    |
    v
[Linear(64->32) + ReLU + Dropout(0.3)]                --> (B, 32)
    |
    v
[Linear(32->8)]                                        --> (B, 8)
```

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath('.'))
from src.focal_loss import FocalLoss, compute_alpha
from src.metrics import plot_confusion_matrix, plot_roc_curves, plot_calibration_curve

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


In [ ]:
df = pd.read_parquet('outputs/mossbauer_processed.parquet')
print(f'Dataset cargado. Shape: {df.shape}')

P_matrix = np.stack(df['P_vec'].values)  # (N, 15)
y_data = df['chem_label'].values          # (N,)

print(f'P_matrix shape: {P_matrix.shape}, y_data shape: {y_data.shape}')

Dataset cargado. Shape: (5029, 19)
P_matrix shape: (5029, 15), y_data shape: (5029,)


In [ ]:
class HyperfineDataset(Dataset):
    def __init__(self, X, y):
        self.features = torch.tensor(X, dtype=torch.float32)
        self.labels = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [ ]:
class LightMLP(nn.Module):
    def __init__(self, input_dim=15, num_classes=8):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
def train_mlp_cascade(epochs=30, lr=0.001, batch_size=64):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof_probs = np.zeros((len(P_matrix), 8))
    oof_preds = np.zeros(len(P_matrix))
    fold_metrics = []
    os.makedirs('outputs/models', exist_ok=True)

    for fold, (train_idx, val_idx) in enumerate(skf.split(P_matrix, y_data)):
        print(f'--- FOLD {fold+1}/5 ---')
        P_train, y_train = P_matrix[train_idx], y_data[train_idx]
        P_val, y_val = P_matrix[val_idx], y_data[val_idx]

        # CRITICO: Z-score solo sobre entrenamiento para evitar fuga de datos
        scaler = StandardScaler()
        P_train_sc = scaler.fit_transform(P_train)
        P_val_sc = scaler.transform(P_val)

        train_ds = HyperfineDataset(P_train_sc, y_train)
        val_ds = HyperfineDataset(P_val_sc, y_val)
        train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
        val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

        model = LightMLP(15, 8).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        alpha = compute_alpha(y_train, num_classes=8).to(device)
        criterion = FocalLoss(alpha=alpha, gamma=2.0)
        best_val_loss = float('inf')

        for epoch in range(epochs):
            model.train()
            for bx, by in train_dl:
                bx, by = bx.to(device), by.to(device)
                optimizer.zero_grad()
                loss = criterion(model(bx), by)
                loss.backward()
                optimizer.step()

            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for bx, by in val_dl:
                    bx, by = bx.to(device), by.to(device)
                    val_loss += criterion(model(bx), by).item() * len(by)
            val_loss /= len(val_ds)
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save(model.state_dict(), f'outputs/models/LightMLP_fold{fold+1}.pt')

        model.load_state_dict(torch.load(f'outputs/models/LightMLP_fold{fold+1}.pt'))
        model.eval()
        val_probs = []
        with torch.no_grad():
            for bx, by in val_dl:
                probs = F.softmax(model(bx.to(device)), dim=-1)
                val_probs.append(probs.cpu().numpy())
        val_probs = np.concatenate(val_probs)
        val_preds = np.argmax(val_probs, axis=1)
        oof_probs[val_idx] = val_probs
        oof_preds[val_idx] = val_preds

        acc = accuracy_score(y_val, val_preds)
        mf1 = f1_score(y_val, val_preds, average='macro')
        wf1 = f1_score(y_val, val_preds, average='weighted')
        print(f'Fold {fold+1} - Acc: {acc:.4f}, Macro-F1: {mf1:.4f}, Weighted-F1: {wf1:.4f}')
        fold_metrics.append([acc, mf1, wf1])

    fm = np.array(fold_metrics)
    means, stds = fm.mean(0), fm.std(0)
    print(f'\n=== LightMLP FINAL ===')
    print(f'Accuracy:    {means[0]:.4f} +/- {stds[0]:.4f}')
    print(f'Macro-F1:    {means[1]:.4f} +/- {stds[1]:.4f}')
    print(f'Weighted-F1: {means[2]:.4f} +/- {stds[2]:.4f}')
    return oof_probs, oof_preds, means, stds

In [ ]:
def evaluate_random_forest():
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof_probs = np.zeros((len(P_matrix), 8))
    oof_preds = np.zeros(len(P_matrix))
    fold_metrics = []
    importances = np.zeros(15)
    feature_labels = ['delta1','Delta1','BHF1','Gamma1','A1',
                      'delta2','Delta2','BHF2','Gamma2','A2',
                      'delta3','Delta3','BHF3','Gamma3','A3']

    for fold, (train_idx, val_idx) in enumerate(skf.split(P_matrix, y_data)):
        P_train, y_train = P_matrix[train_idx], y_data[train_idx]
        P_val, y_val = P_matrix[val_idx], y_data[val_idx]
        rf = RandomForestClassifier(
            n_estimators=300, max_depth=None,
            class_weight='balanced', n_jobs=-1, random_state=42
        )
        rf.fit(P_train, y_train)
        proba = rf.predict_proba(P_val)
        val_probs = np.zeros((len(P_val), 8))
        for i, cls in enumerate(rf.classes_):
            val_probs[:, cls] = proba[:, i]
        val_preds = rf.predict(P_val)
        oof_probs[val_idx] = val_probs
        oof_preds[val_idx] = val_preds
        acc = accuracy_score(y_val, val_preds)
        mf1 = f1_score(y_val, val_preds, average='macro')
        wf1 = f1_score(y_val, val_preds, average='weighted')
        fold_metrics.append([acc, mf1, wf1])
        importances += rf.feature_importances_

    fm = np.array(fold_metrics)
    means, stds = fm.mean(0), fm.std(0)
    importances /= 5.0
    print(f'\n=== Random Forest FINAL ===')
    print(f'Accuracy:    {means[0]:.4f} +/- {stds[0]:.4f}')
    print(f'Macro-F1:    {means[1]:.4f} +/- {stds[1]:.4f}')
    print(f'Weighted-F1: {means[2]:.4f} +/- {stds[2]:.4f}')

    os.makedirs('outputs/results', exist_ok=True)
    idx_sorted = np.argsort(importances)[::-1]
    plt.figure(figsize=(10, 5))
    plt.bar(range(15), importances[idx_sorted], color='teal', edgecolor='black')
    plt.xticks(range(15), [feature_labels[i] for i in idx_sorted], rotation=45, ha='right')
    plt.title('Importancia de Caracteristicas — Random Forest (MDI)')
    plt.xlabel('Parametro Hiperfino')
    plt.ylabel('Importancia Relativa')
    plt.tight_layout()
    plt.savefig('outputs/results/03_rf_feature_importances.png', dpi=150)
    plt.close()
    return oof_probs, oof_preds, means, stds

In [ ]:
print('=== MLP Ligero en Cascada ===')
mlp_probs, mlp_preds, mlp_means, mlp_stds = train_mlp_cascade(epochs=200, lr=0.001)

print('\n=== Random Forest Baseline ===')
rf_probs, rf_preds, rf_means, rf_stds = evaluate_random_forest()

=== MLP Ligero en Cascada ===
--- FOLD 1/5 ---


/home/jd/Projects/MossAI/.venv/lib/python3.12/site-packages/sklearn/model_selection/_split.py:812: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


Fold 1 - Acc: 0.2008, Macro-F1: 0.0833, Weighted-F1: 0.2640
--- FOLD 2/5 ---
Fold 2 - Acc: 0.2078, Macro-F1: 0.1148, Weighted-F1: 0.2644
--- FOLD 3/5 ---
Fold 3 - Acc: 0.1988, Macro-F1: 0.1174, Weighted-F1: 0.2562
--- FOLD 4/5 ---
Fold 4 - Acc: 0.2396, Macro-F1: 0.1032, Weighted-F1: 0.2939
--- FOLD 5/5 ---
Fold 5 - Acc: 0.4448, Macro-F1: 0.1100, Weighted-F1: 0.4367

=== LightMLP FINAL ===
Accuracy:    0.2583 +/- 0.0944
Macro-F1:    0.1057 +/- 0.0122
Weighted-F1: 0.3030 +/- 0.0681

=== Random Forest Baseline ===


/home/jd/Projects/MossAI/.venv/lib/python3.12/site-packages/sklearn/model_selection/_split.py:812: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(



=== Random Forest FINAL ===
Accuracy:    0.5230 +/- 0.0118
Macro-F1:    0.2667 +/- 0.0228
Weighted-F1: 0.5691 +/- 0.0081


In [ ]:
CLASS_NAMES = ['Silicatos', 'Oxidos/Hidroxidos', 'Sulfatos', 'Sulfuros/Teluruos',
               'Oxisales', 'Haluros', 'Metales', 'Amorfos']

os.makedirs('outputs/results', exist_ok=True)
plot_confusion_matrix(y_data, mlp_preds, CLASS_NAMES, 'outputs/results/03_mlp_cascade_confusion_matrix.png')
plot_roc_curves(y_data, mlp_probs, CLASS_NAMES, 'outputs/results/03_mlp_cascade_roc_curves.png')
plot_calibration_curve(y_data, mlp_probs, 'outputs/results/03_mlp_cascade_calibration.png')

plot_confusion_matrix(y_data, rf_preds, CLASS_NAMES, 'outputs/results/03_rf_confusion_matrix.png')
plot_roc_curves(y_data, rf_probs, CLASS_NAMES, 'outputs/results/03_rf_roc_curves.png')
plot_calibration_curve(y_data, rf_probs, 'outputs/results/03_rf_calibration.png')

results_dict = {
    'mlp_cascade_accuracy_mean': float(mlp_means[0]),
    'mlp_cascade_accuracy_std': float(mlp_stds[0]),
    'mlp_cascade_macro_f1_mean': float(mlp_means[1]),
    'mlp_cascade_macro_f1_std': float(mlp_stds[1]),
    'mlp_cascade_weighted_f1_mean': float(mlp_means[2]),
    'mlp_cascade_weighted_f1_std': float(mlp_stds[2]),
    'rf_accuracy_mean': float(rf_means[0]),
    'rf_accuracy_std': float(rf_stds[0]),
    'rf_macro_f1_mean': float(rf_means[1]),
    'rf_macro_f1_std': float(rf_stds[1]),
    'rf_weighted_f1_mean': float(rf_means[2]),
    'rf_weighted_f1_std': float(rf_stds[2])
}
with open('outputs/results/03_cascade_classification_metrics.json', 'w') as f:
    json.dump(results_dict, f, indent=4)

print('Resultados exportados en outputs/results/')

Resultados exportados en outputs/results/
